In [1]:
pip install pandas requests

Note: you may need to restart the kernel to use updated packages.


In [4]:
import requests
import pandas as pd
import time
import os

## Data Extraction

In [5]:
API_URL = "https://archive-api.open-meteo.com/v1/archive"

START_DATE = "2015-01-01"
END_DATE = "2025-12-31"

OUTPUT_FILE = "mumbai_raw_weather_data.csv"

locations = {
    "Bandra": (19.0544, 72.8406),
    "Santacruz": (19.0810, 72.8410),
    "Vile Parle": (19.1000, 72.8430),
    "Andheri": (19.1197, 72.8464),
    "Jogeshwari": (19.1370, 72.8490),
    "Goregaon": (19.1663, 72.8526),
    "Malad": (19.1874, 72.8484),
    "Kandivali": (19.2045, 72.8510),
    "Borivali": (19.2307, 72.8567)
}

hourly_variables = [
    "precipitation",
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "weather_code"
]

# -----------------------------------
# 1. LOAD EXISTING DATA
# -----------------------------------

if os.path.exists(OUTPUT_FILE):
    final_df = pd.read_csv(OUTPUT_FILE)

    completed_locations = set(final_df["zone_name"].unique())

    print("Existing CSV loaded.")
    print("Already extracted:", completed_locations)

else:
    final_df = pd.DataFrame()
    completed_locations = set()

# -----------------------------------
# 2. EXTRACT REMAINING LOCATIONS
# -----------------------------------

session = requests.Session()

for zone_name, (latitude, longitude) in locations.items():

    if zone_name in completed_locations:
        print(f"\nSkipping {zone_name}: Already extracted.")
        continue

    print(f"\nExtracting {zone_name}...")

    params = {
        "latitude": latitude,
        "longitude": longitude,
        "start_date": START_DATE,
        "end_date": END_DATE,
        "hourly": ",".join(hourly_variables),
        "timezone": "Asia/Kolkata",
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm"
    }

    success = False

    # -----------------------------------
    # 3. RETRY MECHANISM
    # -----------------------------------

    for attempt in range(1, 7):

        try:
            response = session.get(
                API_URL,
                params=params,
                timeout=180
            )

            if response.status_code == 429:
                retry_after = response.headers.get("Retry-After")

                if retry_after and retry_after.isdigit():
                    wait_time = int(retry_after)
                else:
                    wait_time = min(30 * (2 ** (attempt - 1)), 300)

                print(f"Rate limited. Waiting {wait_time} seconds...")
                time.sleep(wait_time)
                continue

            response.raise_for_status()

            data = response.json()

            hourly_data = data.get("hourly")

            if not hourly_data:
                print("No hourly data returned.")
                break

            df = pd.DataFrame(hourly_data)

            df.rename(
                columns={
                    "time": "timestamp",
                    "precipitation": "rainfall_1hr_mm"
                },
                inplace=True
            )

            df["zone_name"] = zone_name
            df["latitude"] = latitude
            df["longitude"] = longitude

            # -----------------------------------
            # 4. APPEND AND SAVE
            # -----------------------------------

            final_df = pd.concat(
                [final_df, df],
                ignore_index=True
            )

            final_df.to_csv(
                OUTPUT_FILE,
                index=False
            )

            print(f"Successfully extracted {len(df)} records.")

            success = True
            completed_locations.add(zone_name)

            # Wait before next location
            time.sleep(15)
            break

        except requests.exceptions.RequestException as error:

            wait_time = min(15 * (2 ** (attempt - 1)), 120)

            print(f"Attempt {attempt} failed: {error}")

            if attempt < 6:
                print(f"Retrying in {wait_time} seconds...")
                time.sleep(wait_time)

    if not success:
        print(f"FAILED: {zone_name}. You can retry it later.")

    print("Current total records:", len(final_df))

# -----------------------------------
# 5. FINAL STATUS
# -----------------------------------

print("\n========== EXTRACTION SUMMARY ==========")

print("Successful locations:", sorted(completed_locations))

remaining = set(locations.keys()) - completed_locations

print("Remaining locations:", sorted(remaining))

print("Total records:", len(final_df))

if not remaining:
    print("ALL 9 LOCATIONS EXTRACTED SUCCESSFULLY!")
else:
    print("Extraction partially completed. Run the script again later.")

print("CSV file:", OUTPUT_FILE)


Existing CSV loaded.
Already extracted: {'Santacruz', 'Andheri', 'Jogeshwari', 'Vile Parle', 'Bandra'}

Skipping Bandra: Already extracted.

Skipping Santacruz: Already extracted.

Skipping Vile Parle: Already extracted.

Skipping Andheri: Already extracted.

Skipping Jogeshwari: Already extracted.

Extracting Goregaon...
Successfully extracted 96432 records.
Current total records: 578592

Extracting Malad...
Successfully extracted 96432 records.
Current total records: 675024

Extracting Kandivali...
Successfully extracted 96432 records.
Current total records: 771456

Extracting Borivali...
Successfully extracted 96432 records.
Current total records: 867888

========== EXTRACTION SUMMARY ==========
Successful locations: ['Andheri', 'Bandra', 'Borivali', 'Goregaon', 'Jogeshwari', 'Kandivali', 'Malad', 'Santacruz', 'Vile Parle']
Remaining locations: []
Total records: 867888
ALL 9 LOCATIONS EXTRACTED SUCCESSFULLY!
CSV file: mumbai_raw_weather_data.csv


## Data Validation

In [8]:
import pandas as pd

In [9]:


# Load raw dataset
df = pd.read_csv("mumbai_raw_weather_data.csv")

# Convert timestamp
df["timestamp"] = pd.to_datetime(df["timestamp"])

# 1. Dataset overview
print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

# 2. Records per location
print("\nRecords per location:")
print(df.groupby("zone_name").size())

# 3. Missing values
print("\nMissing values:")
print(df.isnull().sum())

# 4. Duplicate records
print("\nDuplicate rows:", df.duplicated().sum())

# 5. Duplicate timestamps within locations
print(
    "Duplicate location-timestamp pairs:",
    df.duplicated(subset=["zone_name", "timestamp"]).sum()
)

# 6. Rainfall statistics
print("\nRainfall statistics:")
print(df["rainfall_1hr_mm"].describe())

# 7. Date range
print("\nDate range:")
print(df["timestamp"].min(), "to", df["timestamp"].max())

Dataset shape: (867888, 9)

Columns:
['timestamp', 'rainfall_1hr_mm', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'weather_code', 'zone_name', 'latitude', 'longitude']

Records per location:
zone_name
Andheri       96432
Bandra        96432
Borivali      96432
Goregaon      96432
Jogeshwari    96432
Kandivali     96432
Malad         96432
Santacruz     96432
Vile Parle    96432
dtype: int64

Missing values:
timestamp               0
rainfall_1hr_mm         0
temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
weather_code            0
zone_name               0
latitude                0
longitude               0
dtype: int64

Duplicate rows: 0
Duplicate location-timestamp pairs: 0

Rainfall statistics:
count    867888.000000
mean          0.253684
std           1.012030
min           0.000000
25%           0.000000
50%           0.000000
75%           0.100000
max          76.100000
Name: rainfall_1hr_mm, dtype: float64

Date range:
2015-01-01 0

# Feature Engineering & Dataset Creation

In [10]:
import pandas as pd

In [11]:
# -----------------------------------------
# 1. Load raw dataset
# -----------------------------------------

df = pd.read_csv("mumbai_raw_weather_data.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])

# Sort by location and timestamp
df = df.sort_values(
    by=["zone_name", "timestamp"]
).reset_index(drop=True)

# Remove duplicate location-timestamp records
df = df.drop_duplicates(
    subset=["zone_name", "timestamp"]
)

# -----------------------------------------
# 2. Calculate rainfall features
# -----------------------------------------

feature_data = []

for zone, group in df.groupby("zone_name"):

    print(f"Processing {zone}...")

    group = group.sort_values("timestamp").set_index("timestamp")

    # Create a complete hourly timeline.
    # Missing hours remain NaN; they are NOT treated as zero.
    rainfall = group["rainfall_1hr_mm"].asfreq("h")

    features = pd.DataFrame(index=rainfall.index)

    # Previous 3 hours
    features["rainfall_3hr_mm"] = (
        rainfall.rolling(window=3, min_periods=3).sum()
    )

    # Previous 6 hours
    features["rainfall_6hr_mm"] = (
        rainfall.rolling(window=6, min_periods=6).sum()
    )

    # Previous 24 hours
    features["rainfall_24hr_mm"] = (
        rainfall.rolling(window=24, min_periods=24).sum()
    )

    # Previous 48 hours
    features["rainfall_48hr_mm"] = (
        rainfall.rolling(window=48, min_periods=48).sum()
    )

    features["zone_name"] = zone

    feature_data.append(features)

# -----------------------------------------
# 3. Combine calculated features
# -----------------------------------------

features_df = pd.concat(feature_data)

# Join derived features with original dataset
df = df.set_index(["zone_name", "timestamp"])

features_df = features_df.reset_index().set_index(
    ["zone_name", "timestamp"]
)

df = df.join(features_df)

df = df.reset_index()

# -----------------------------------------
# 4. Save cleaned dataset
# -----------------------------------------

df.to_csv("mumbai_clean_weather_data.csv", index=False)

print("\n========== FEATURE ENGINEERING SUMMARY ==========")

print("Total records:", len(df))
print("Total locations:", df["zone_name"].nunique())

print("\nNew rainfall features:")
print([
    "rainfall_3hr_mm",
    "rainfall_6hr_mm",
    "rainfall_24hr_mm",
    "rainfall_48hr_mm"
])

print("\nMissing values in new features:")
print(df[
    [
        "rainfall_3hr_mm",
        "rainfall_6hr_mm",
        "rainfall_24hr_mm",
        "rainfall_48hr_mm"
    ]
].isnull().sum())

print("\nCleaned dataset saved as:")
print("mumbai_clean_weather_data.csv")

Processing Andheri...
Processing Bandra...
Processing Borivali...
Processing Goregaon...
Processing Jogeshwari...
Processing Kandivali...
Processing Malad...
Processing Santacruz...
Processing Vile Parle...

========== FEATURE ENGINEERING SUMMARY ==========
Total records: 867888
Total locations: 9

New rainfall features:
['rainfall_3hr_mm', 'rainfall_6hr_mm', 'rainfall_24hr_mm', 'rainfall_48hr_mm']

Missing values in new features:
rainfall_3hr_mm      18
rainfall_6hr_mm      45
rainfall_24hr_mm    207
rainfall_48hr_mm    423
dtype: int64

Cleaned dataset saved as:
mumbai_clean_weather_data.csv


## Final Dataset Validation

In [12]:
import pandas as pd

df = pd.read_csv("mumbai_clean_weather_data.csv")

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nRecords per location:")
print(df.groupby("zone_name").size())

print("\nDuplicate records:")
print(df.duplicated().sum())

Dataset shape: (867888, 13)

Columns:
['zone_name', 'timestamp', 'rainfall_1hr_mm', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'weather_code', 'latitude', 'longitude', 'rainfall_3hr_mm', 'rainfall_6hr_mm', 'rainfall_24hr_mm', 'rainfall_48hr_mm']

Missing values:
zone_name                 0
timestamp                 0
rainfall_1hr_mm           0
temperature_2m            0
relative_humidity_2m      0
wind_speed_10m            0
weather_code              0
latitude                  0
longitude                 0
rainfall_3hr_mm          18
rainfall_6hr_mm          45
rainfall_24hr_mm        207
rainfall_48hr_mm        423
dtype: int64

Records per location:
zone_name
Andheri       96432
Bandra        96432
Borivali      96432
Goregaon      96432
Jogeshwari    96432
Kandivali     96432
Malad         96432
Santacruz     96432
Vile Parle    96432
dtype: int64

Duplicate records:
0


# Inspect the cleaned weather dataset

In [1]:
import pandas as pd

weather_df = pd.read_csv("mumbai_clean_weather_data.csv")

print("Dataset shape:", weather_df.shape)

print("\nColumns:")
print(weather_df.columns.tolist())

print("\nData types:")
print(weather_df.dtypes)

print("\nMissing values:")
print(weather_df.isnull().sum())

print("\nFirst 5 records:")
display(weather_df.head())

Dataset shape: (867888, 13)

Columns:
['zone_name', 'timestamp', 'rainfall_1hr_mm', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'weather_code', 'latitude', 'longitude', 'rainfall_3hr_mm', 'rainfall_6hr_mm', 'rainfall_24hr_mm', 'rainfall_48hr_mm']

Data types:
zone_name                object
timestamp                object
rainfall_1hr_mm         float64
temperature_2m          float64
relative_humidity_2m      int64
wind_speed_10m          float64
weather_code              int64
latitude                float64
longitude               float64
rainfall_3hr_mm         float64
rainfall_6hr_mm         float64
rainfall_24hr_mm        float64
rainfall_48hr_mm        float64
dtype: object

Missing values:
zone_name                 0
timestamp                 0
rainfall_1hr_mm           0
temperature_2m            0
relative_humidity_2m      0
wind_speed_10m            0
weather_code              0
latitude                  0
longitude                 0
rainfall_3hr_mm          

,zone_name,timestamp,rainfall_1hr_mm,temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code,latitude,longitude,rainfall_3hr_mm,rainfall_6hr_mm,rainfall_24hr_mm,rainfall_48hr_mm
0,Andheri,2015-01-01 00:00:00,0.0,21.4,72,8.2,2,19.1197,72.8464,NaN,NaN,NaN,NaN
1,Andheri,2015-01-01 01:00:00,0.0,21.1,73,7.9,2,19.1197,72.8464,NaN,NaN,NaN,NaN
2,Andheri,2015-01-01 02:00:00,0.0,20.7,75,8.5,2,19.1197,72.8464,0.0,NaN,NaN,NaN
3,Andheri,2015-01-01 03:00:00,0.0,20.3,77,7.4,2,19.1197,72.8464,0.0,NaN,NaN,NaN
4,Andheri,2015-01-01 04:00:00,0.0,19.9,79,6.9,1,19.1197,72.8464,0.0,NaN,NaN,NaN


# Validate weather coverage and timestamps

In [2]:
# Check locality coverage
print("Records per locality:")
display(weather_df["zone_name"].value_counts().sort_index())

# Convert timestamp to datetime
weather_df["timestamp"] = pd.to_datetime(
    weather_df["timestamp"],
    errors="coerce"
)

print("\nTimestamp range:")
print("Start:", weather_df["timestamp"].min())
print("End:", weather_df["timestamp"].max())

# Check unique timestamps per locality
print("\nUnique timestamps per locality:")
display(
    weather_df.groupby("zone_name")["timestamp"]
    .nunique()
)

# Check duplicate locality-timestamp combinations
duplicates = weather_df.duplicated(
    subset=["zone_name", "timestamp"]
).sum()

print("\nDuplicate locality-timestamp records:", duplicates)

# Check missing values in rainfall accumulation features
rainfall_columns = [
    "rainfall_1hr_mm",
    "rainfall_3hr_mm",
    "rainfall_6hr_mm",
    "rainfall_24hr_mm",
    "rainfall_48hr_mm"
]

print("\nMissing rainfall values by locality:")
display(
    weather_df.groupby("zone_name")[rainfall_columns]
    .apply(lambda x: x.isna().sum())
)

Records per locality:


zone_name
Andheri       96432
Bandra        96432
Borivali      96432
Goregaon      96432
Jogeshwari    96432
Kandivali     96432
Malad         96432
Santacruz     96432
Vile Parle    96432
Name: count, dtype: int64


Timestamp range:
Start: 2015-01-01 00:00:00
End: 2025-12-31 23:00:00

Unique timestamps per locality:


zone_name
Andheri       96432
Bandra        96432
Borivali      96432
Goregaon      96432
Jogeshwari    96432
Kandivali     96432
Malad         96432
Santacruz     96432
Vile Parle    96432
Name: timestamp, dtype: int64


Duplicate locality-timestamp records: 0

Missing rainfall values by locality:


,rainfall_1hr_mm,rainfall_3hr_mm,rainfall_6hr_mm,rainfall_24hr_mm,rainfall_48hr_mm
zone_name,,,,,
Andheri,0,2,5,23,47
Bandra,0,2,5,23,47
Borivali,0,2,5,23,47
Goregaon,0,2,5,23,47
Jogeshwari,0,2,5,23,47
Kandivali,0,2,5,23,47
Malad,0,2,5,23,47
Santacruz,0,2,5,23,47
Vile Parle,0,2,5,23,47


# Verify missing-value positions

In [3]:
# Check whether missing rainfall values occur only at the beginning

rainfall_columns = [
    "rainfall_3hr_mm",
    "rainfall_6hr_mm",
    "rainfall_24hr_mm",
    "rainfall_48hr_mm"
]

for locality, group in weather_df.groupby("zone_name"):

    group = group.sort_values("timestamp").reset_index(drop=True)

    print(f"\n{locality}")

    for col in rainfall_columns:

        missing_indices = group.index[group[col].isna()]

        if len(missing_indices) > 0:
            print(
                col,
                "| First missing index:", missing_indices.min(),
                "| Last missing index:", missing_indices.max(),
                "| Missing count:", len(missing_indices)
            )

        else:
            print(col, "| No missing values")

# Verify hourly continuity
print("\nTimestamp interval validation:")

for locality, group in weather_df.groupby("zone_name"):

    intervals = (
        group.sort_values("timestamp")["timestamp"]
        .diff()
        .dropna()
    )

    print(
        locality,
        "| Non-hourly intervals:",
        (intervals != pd.Timedelta(hours=1)).sum()
    )


Andheri
rainfall_3hr_mm | First missing index: 0 | Last missing index: 1 | Missing count: 2
rainfall_6hr_mm | First missing index: 0 | Last missing index: 4 | Missing count: 5
rainfall_24hr_mm | First missing index: 0 | Last missing index: 22 | Missing count: 23
rainfall_48hr_mm | First missing index: 0 | Last missing index: 46 | Missing count: 47

Bandra
rainfall_3hr_mm | First missing index: 0 | Last missing index: 1 | Missing count: 2
rainfall_6hr_mm | First missing index: 0 | Last missing index: 4 | Missing count: 5
rainfall_24hr_mm | First missing index: 0 | Last missing index: 22 | Missing count: 23
rainfall_48hr_mm | First missing index: 0 | Last missing index: 46 | Missing count: 47

Borivali
rainfall_3hr_mm | First missing index: 0 | Last missing index: 1 | Missing count: 2
rainfall_6hr_mm | First missing index: 0 | Last missing index: 4 | Missing count: 5
rainfall_24hr_mm | First missing index: 0 | Last missing index: 22 | Missing count: 23
rainfall_48hr_mm | First missing i

# Clean the initial rolling-window records

In [4]:
# Remove initial records where 48-hour rainfall is unavailable

weather_df = weather_df.sort_values(
    ["zone_name", "timestamp"]
).reset_index(drop=True)

before = len(weather_df)

weather_df = weather_df.dropna(
    subset=["rainfall_48hr_mm"]
).reset_index(drop=True)

after = len(weather_df)

print("Records before cleaning:", before)
print("Records after cleaning:", after)
print("Records removed:", before - after)

print("\nRemaining missing values:")
display(weather_df.isnull().sum())

print("\nRecords per locality:")
display(weather_df["zone_name"].value_counts().sort_index())

Records before cleaning: 867888
Records after cleaning: 867465
Records removed: 423

Remaining missing values:


zone_name               0
timestamp               0
rainfall_1hr_mm         0
temperature_2m          0
relative_humidity_2m    0
wind_speed_10m          0
weather_code            0
latitude                0
longitude               0
rainfall_3hr_mm         0
rainfall_6hr_mm         0
rainfall_24hr_mm        0
rainfall_48hr_mm        0
dtype: int64


Records per locality:


zone_name
Andheri       96385
Bandra        96385
Borivali      96385
Goregaon      96385
Jogeshwari    96385
Kandivali     96385
Malad         96385
Santacruz     96385
Vile Parle    96385
Name: count, dtype: int64

# Save the cleaned weather dataset

In [5]:
# Save cleaned weather dataset

output_file = "mumbai_clean_weather_data_final.csv"

weather_df.to_csv(output_file, index=False)

print("Cleaned weather dataset saved:", output_file)

# Verify exported file
verified_weather = pd.read_csv(output_file)

print("Exported shape:", verified_weather.shape)
print("Missing values:", verified_weather.isnull().sum().sum())

Cleaned weather dataset saved: mumbai_clean_weather_data_final.csv
Exported shape: (867465, 13)
Missing values: 0


# Integrate weather and terrain datasets

In [6]:
import pandas as pd

# Load cleaned weather dataset
weather_df = pd.read_csv("mumbai_clean_weather_data_final.csv")

# Load terrain dataset
terrain_df = pd.read_csv("mumbai_terrain_features.csv")

# Keep terrain reference coordinates separate from weather coordinates
terrain_df = terrain_df.rename(columns={
    "latitude": "terrain_reference_latitude",
    "longitude": "terrain_reference_longitude"
})

# Merge terrain features into hourly weather records
combined_df = weather_df.merge(
    terrain_df,
    on="zone_name",
    how="left",
    validate="many_to_one"
)

# Validate integration
print("Weather dataset shape:", weather_df.shape)
print("Terrain dataset shape:", terrain_df.shape)
print("Combined dataset shape:", combined_df.shape)

print("\nMissing values:")
display(combined_df.isnull().sum())

print("\nRecords per locality:")
display(combined_df["zone_name"].value_counts().sort_index())

print("\nCombined dataset preview:")
display(combined_df.head())

Weather dataset shape: (867465, 13)
Terrain dataset shape: (9, 7)
Combined dataset shape: (867465, 19)

Missing values:


zone_name                         0
timestamp                         0
rainfall_1hr_mm                   0
temperature_2m                    0
relative_humidity_2m              0
wind_speed_10m                    0
weather_code                      0
latitude                          0
longitude                         0
rainfall_3hr_mm                   0
rainfall_6hr_mm                   0
rainfall_24hr_mm                  0
rainfall_48hr_mm                  0
zone_id                           0
terrain_reference_latitude        0
terrain_reference_longitude       0
elevation_m                       0
distance_to_coast_m               0
distance_to_nearest_waterway_m    0
dtype: int64


Records per locality:


zone_name
Andheri       96385
Bandra        96385
Borivali      96385
Goregaon      96385
Jogeshwari    96385
Kandivali     96385
Malad         96385
Santacruz     96385
Vile Parle    96385
Name: count, dtype: int64


Combined dataset preview:


,zone_name,timestamp,rainfall_1hr_mm,temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code,latitude,longitude,rainfall_3hr_mm,rainfall_6hr_mm,rainfall_24hr_mm,rainfall_48hr_mm,zone_id,terrain_reference_latitude,terrain_reference_longitude,elevation_m,distance_to_coast_m,distance_to_nearest_waterway_m
0,Andheri,2015-01-02 23:00:00,0.0,20.9,86,7.6,0,19.1197,72.8464,0.0,0.0,0.0,0.0,3,19.12,72.85,19.0,3021.011743,755.981633
1,Andheri,2015-01-03 00:00:00,0.0,20.4,89,7.2,0,19.1197,72.8464,0.0,0.0,0.0,0.0,3,19.12,72.85,19.0,3021.011743,755.981633
2,Andheri,2015-01-03 01:00:00,0.0,19.9,91,6.3,0,19.1197,72.8464,0.0,0.0,0.0,0.0,3,19.12,72.85,19.0,3021.011743,755.981633
3,Andheri,2015-01-03 02:00:00,0.0,19.6,92,6.3,0,19.1197,72.8464,0.0,0.0,0.0,0.0,3,19.12,72.85,19.0,3021.011743,755.981633
4,Andheri,2015-01-03 03:00:00,0.0,19.2,93,7.4,0,19.1197,72.8464,0.0,0.0,0.0,0.0,3,19.12,72.85,19.0,3021.011743,755.981633


# Save the integrated dataset

In [7]:
# Save the integrated weather + terrain dataset
output_file = "mumbai_weather_terrain_combined.csv"

combined_df.to_csv(output_file, index=False)

# Verify the saved file
verified_df = pd.read_csv(output_file)

print("Saved file:", output_file)
print("Verified shape:", verified_df.shape)
print("Total missing values:", verified_df.isnull().sum().sum())
print("Unique localities:", verified_df["zone_name"].nunique())
print("Records per locality:")
display(verified_df["zone_name"].value_counts().sort_index())

Saved file: mumbai_weather_terrain_combined.csv
Verified shape: (867465, 19)
Total missing values: 0
Unique localities: 9
Records per locality:


zone_name
Andheri       96385
Bandra        96385
Borivali      96385
Goregaon      96385
Jogeshwari    96385
Kandivali     96385
Malad         96385
Santacruz     96385
Vile Parle    96385
Name: count, dtype: int64

# Merge BMC hotspot features with weather + terrain

In [8]:
import pandas as pd

# Load integrated weather + terrain dataset
combined_df = pd.read_csv("mumbai_weather_terrain_combined.csv")

# Load locality-wise BMC hotspot summary
hotspot_df = pd.read_csv("mumbai_locality_hotspot_summary.csv")

# Merge using locality name
final_df = combined_df.merge(
    hotspot_df,
    on="zone_name",
    how="left",
    validate="many_to_one"
)

# Validate integration
print("Weather + terrain shape:", combined_df.shape)
print("Hotspot summary shape:", hotspot_df.shape)
print("Final integrated shape:", final_df.shape)

print("\nMissing values:")
display(final_df.isnull().sum())

print("\nHotspot counts by locality:")
display(
    final_df[
        ["zone_name", "chronic_flooding_spot_count",
         "flooding_spot_count", "total_hotspot_count"]
    ].drop_duplicates().sort_values("zone_name")
)

print("\nRecords per locality:")
display(final_df["zone_name"].value_counts().sort_index())

Weather + terrain shape: (867465, 19)
Hotspot summary shape: (9, 4)
Final integrated shape: (867465, 22)

Missing values:


zone_name                         0
timestamp                         0
rainfall_1hr_mm                   0
temperature_2m                    0
relative_humidity_2m              0
wind_speed_10m                    0
weather_code                      0
latitude                          0
longitude                         0
rainfall_3hr_mm                   0
rainfall_6hr_mm                   0
rainfall_24hr_mm                  0
rainfall_48hr_mm                  0
zone_id                           0
terrain_reference_latitude        0
terrain_reference_longitude       0
elevation_m                       0
distance_to_coast_m               0
distance_to_nearest_waterway_m    0
chronic_flooding_spot_count       0
flooding_spot_count               0
total_hotspot_count               0
dtype: int64


Hotspot counts by locality:


,zone_name,chronic_flooding_spot_count,flooding_spot_count,total_hotspot_count
0,Andheri,9,1,10
96385,Bandra,4,2,6
192770,Borivali,6,1,7
289155,Goregaon,8,0,8
385540,Jogeshwari,3,0,3
481925,Kandivali,7,0,7
578310,Malad,10,3,13
674695,Santacruz,4,1,5
771080,Vile Parle,2,0,2



Records per locality:


zone_name
Andheri       96385
Bandra        96385
Borivali      96385
Goregaon      96385
Jogeshwari    96385
Kandivali     96385
Malad         96385
Santacruz     96385
Vile Parle    96385
Name: count, dtype: int64

# Save the integrated dataset

In [9]:
# Save final weather + terrain + BMC hotspot dataset
output_file = "mumbai_flood_model_features.csv"

final_df.to_csv(output_file, index=False)

# Verify saved dataset
verified_df = pd.read_csv(output_file)

print("Saved file:", output_file)
print("Verified shape:", verified_df.shape)
print("Total missing values:", verified_df.isnull().sum().sum())
print("Unique localities:", verified_df["zone_name"].nunique())
print("Unique timestamps:", verified_df["timestamp"].nunique())

print("\nRecords per locality:")
display(verified_df["zone_name"].value_counts().sort_index())

Saved file: mumbai_flood_model_features.csv
Verified shape: (867465, 22)
Total missing values: 0
Unique localities: 9
Unique timestamps: 96385

Records per locality:


zone_name
Andheri       96385
Bandra        96385
Borivali      96385
Goregaon      96385
Jogeshwari    96385
Kandivali     96385
Malad         96385
Santacruz     96385
Vile Parle    96385
Name: count, dtype: int64

# validate the final model-feature dataset

In [10]:
import pandas as pd

features_df = pd.read_csv("mumbai_flood_model_features.csv")

print("Dataset shape:", features_df.shape)

print("\nColumn data types:")
display(features_df.dtypes.to_frame("data_type"))

print("\nRecords per locality:")
display(features_df["zone_name"].value_counts().sort_index())

print("\nTimestamp range:")
print("Start:", features_df["timestamp"].min())
print("End:", features_df["timestamp"].max())

print("\nDuplicate locality-timestamp records:",
      features_df.duplicated(["zone_name", "timestamp"]).sum())

Dataset shape: (867465, 22)

Column data types:


,data_type
zone_name,object
timestamp,object
rainfall_1hr_mm,float64
temperature_2m,float64
relative_humidity_2m,int64
wind_speed_10m,float64
weather_code,int64
latitude,float64
longitude,float64
rainfall_3hr_mm,float64



Records per locality:


zone_name
Andheri       96385
Bandra        96385
Borivali      96385
Goregaon      96385
Jogeshwari    96385
Kandivali     96385
Malad         96385
Santacruz     96385
Vile Parle    96385
Name: count, dtype: int64


Timestamp range:
Start: 2015-01-02 23:00:00
End: 2025-12-31 23:00:00

Duplicate locality-timestamp records: 0


# convert timestamp and verify

In [11]:
# Convert timestamp to datetime
features_df["timestamp"] = pd.to_datetime(
    features_df["timestamp"],
    errors="coerce"
)

# Verify conversion
print("Timestamp data type:", features_df["timestamp"].dtype)
print("Invalid timestamps:", features_df["timestamp"].isna().sum())

# Extract useful time-based features
features_df["hour"] = features_df["timestamp"].dt.hour
features_df["month"] = features_df["timestamp"].dt.month
features_df["day_of_week"] = features_df["timestamp"].dt.dayofweek

display(features_df[
    ["zone_name", "timestamp", "hour", "month", "day_of_week"]
].head())

Timestamp data type: datetime64[ns]
Invalid timestamps: 0


,zone_name,timestamp,hour,month,day_of_week
0,Andheri,2015-01-02 23:00:00,23,1,4
1,Andheri,2015-01-03 00:00:00,0,1,5
2,Andheri,2015-01-03 01:00:00,1,1,5
3,Andheri,2015-01-03 02:00:00,2,1,5
4,Andheri,2015-01-03 03:00:00,3,1,5


# add monsoon-season feature

In [12]:
# Identify monsoon season (June to September)
features_df["is_monsoon"] = features_df["month"].isin(
    [6, 7, 8, 9]
).astype(int)

# Verify distribution
print("Monsoon feature distribution:")
display(
    features_df["is_monsoon"].value_counts()
)

print("\nMonsoon records by locality:")
display(
    features_df.groupby("zone_name")["is_monsoon"]
    .value_counts()
    .unstack(fill_value=0)
)

Monsoon feature distribution:


is_monsoon
0    577593
1    289872
Name: count, dtype: int64


Monsoon records by locality:


is_monsoon,0,1
zone_name,,
Andheri,64177,32208
Bandra,64177,32208
Borivali,64177,32208
Goregaon,64177,32208
Jogeshwari,64177,32208
Kandivali,64177,32208
Malad,64177,32208
Santacruz,64177,32208
Vile Parle,64177,32208


# save the temporally enriched dataset

In [13]:
# Save dataset with engineered temporal features
features_df.to_csv(
    "mumbai_flood_model_features_temporal.csv",
    index=False
)

print("Temporally enriched dataset saved successfully.")
print("Shape:", features_df.shape)
print("Columns:", features_df.columns.tolist())

Temporally enriched dataset saved successfully.
Shape: (867465, 26)
Columns: ['zone_name', 'timestamp', 'rainfall_1hr_mm', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'weather_code', 'latitude', 'longitude', 'rainfall_3hr_mm', 'rainfall_6hr_mm', 'rainfall_24hr_mm', 'rainfall_48hr_mm', 'zone_id', 'terrain_reference_latitude', 'terrain_reference_longitude', 'elevation_m', 'distance_to_coast_m', 'distance_to_nearest_waterway_m', 'chronic_flooding_spot_count', 'flooding_spot_count', 'total_hotspot_count', 'hour', 'month', 'day_of_week', 'is_monsoon']


# create the alternative training dataset

In [14]:
import pandas as pd

df = pd.read_csv("mumbai_flood_model_features_temporal.csv")

df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.sort_values(["zone_name", "timestamp"]).reset_index(drop=True)

# Calculate rainfall over the NEXT 24 hours
def calculate_future_rainfall(series):
    return (
        series.iloc[::-1]
        .shift(1)
        .rolling(24, min_periods=24)
        .sum()
        .iloc[::-1]
    )

df["future_24h_rainfall_mm"] = (
    df.groupby("zone_name")["rainfall_1hr_mm"]
    .transform(calculate_future_rainfall)
)

# IMD heavy rainfall threshold
df["heavy_rainfall_next_24h"] = (
    df["future_24h_rainfall_mm"] >= 64.5
).astype("Int64")

# Remove rows without a complete future 24-hour window
df = df.dropna(subset=["future_24h_rainfall_mm"]).copy()

# Keep original features and the target only
df = df.drop(columns=["future_24h_rainfall_mm"])

df.to_csv("mumbai_heavy_rainfall_forecast_dataset.csv", index=False)

print("Dataset shape:", df.shape)
print("\nTarget distribution:")
print(df["heavy_rainfall_next_24h"].value_counts())

print("\nLocality distribution:")
print(df["zone_name"].value_counts())

Dataset shape: (867249, 27)

Target distribution:
heavy_rainfall_next_24h
0    854388
1     12861
Name: count, dtype: Int64

Locality distribution:
zone_name
Andheri       96361
Bandra        96361
Borivali      96361
Goregaon      96361
Jogeshwari    96361
Kandivali     96361
Malad         96361
Santacruz     96361
Vile Parle    96361
Name: count, dtype: int64


# validate positive cases by locality and year

In [15]:
df = pd.read_csv("mumbai_heavy_rainfall_forecast_dataset.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

print("POSITIVE CASES BY LOCALITY")
print(
    df.groupby("zone_name")["heavy_rainfall_next_24h"]
      .agg(["sum", "count"])
      .assign(
          positive_percentage=lambda x: (x["sum"] / x["count"] * 100).round(2)
      )
)

print("\nPOSITIVE CASES BY YEAR")
df["year"] = df["timestamp"].dt.year
print(
    df.groupby("year")["heavy_rainfall_next_24h"]
      .agg(["sum", "count"])
)

print("\nTARGET MISSING VALUES:",
      df["heavy_rainfall_next_24h"].isna().sum())

POSITIVE CASES BY LOCALITY
             sum  count  positive_percentage
zone_name                                   
Andheri     1541  96361                 1.60
Bandra      1541  96361                 1.60
Borivali    1340  96361                 1.39
Goregaon    1339  96361                 1.39
Jogeshwari  1339  96361                 1.39
Kandivali   1340  96361                 1.39
Malad       1339  96361                 1.39
Santacruz   1541  96361                 1.60
Vile Parle  1541  96361                 1.60

POSITIVE CASES BY YEAR
       sum  count
year             
2015   399  78417
2016  1275  79056
2017  1527  78840
2018   108  78840
2019  1585  78840
2020  1669  79056
2021  1176  78840
2022  1419  78840
2023   717  78840
2024  1551  79056
2025  1435  78624

TARGET MISSING VALUES: 0


In [16]:
import pandas as pd
import hashlib

df = pd.read_csv("mumbai_heavy_rainfall_forecast_dataset.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Check whether localities have identical hourly rainfall sequences
fingerprints = {}

for zone, group in df.groupby("zone_name"):
    rainfall_hash = pd.util.hash_pandas_object(
        group.sort_values("timestamp")["rainfall_1hr_mm"],
        index=False
    ).values.tobytes()

    fingerprints[zone] = hashlib.sha256(rainfall_hash).hexdigest()

print("UNIQUE RAINFALL SEQUENCES:",
      len(set(fingerprints.values())))

print("\nLOCALITY SEQUENCE COMPARISON:")
for zone, fingerprint in fingerprints.items():
    matching = [
        other for other, other_fp in fingerprints.items()
        if other_fp == fingerprint and other != zone
    ]
    print(zone, "matches:", matching)

# Check the unusually low 2018 positive count
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month

print("\n2018 MONTHLY RECORDS AND POSITIVE CASES:")
print(
    df[df["year"] == 2018]
    .groupby("month")["heavy_rainfall_next_24h"]
    .agg(["count", "sum"])
)

print("\nALL-YEAR MONTHLY POSITIVE CASES:")
print(
    df.groupby(["year", "month"])["heavy_rainfall_next_24h"]
    .sum()
    .unstack(fill_value=0)
)

UNIQUE RAINFALL SEQUENCES: 3

LOCALITY SEQUENCE COMPARISON:
Andheri matches: ['Bandra', 'Santacruz', 'Vile Parle']
Bandra matches: ['Andheri', 'Santacruz', 'Vile Parle']
Borivali matches: ['Kandivali']
Goregaon matches: ['Jogeshwari', 'Malad']
Jogeshwari matches: ['Goregaon', 'Malad']
Kandivali matches: ['Borivali']
Malad matches: ['Goregaon', 'Jogeshwari']
Santacruz matches: ['Andheri', 'Bandra', 'Vile Parle']
Vile Parle matches: ['Andheri', 'Bandra', 'Santacruz']

2018 MONTHLY RECORDS AND POSITIVE CASES:
       count  sum
month            
1       6696    0
2       6048    0
3       6696    0
4       6480    0
5       6696    0
6       6480  108
7       6696    0
8       6696    0
9       6480    0
10      6696    0
11      6480    0
12      6696    0

ALL-YEAR MONTHLY POSITIVE CASES:
month  1   2   3   4    5    6     7    8    9    10  11   12
year                                                         
2015    0   0   0   0    0  399     0    0    0    0   0    0
2016    0   0   

In [17]:
import pandas as pd

df = pd.read_csv("mumbai_flood_model_features_temporal.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

# Check coordinates and rainfall variation
print("LOCALITY COORDINATES AND RAINFALL SUMMARY:")
print(
    df.groupby("zone_name").agg(
        latitude=("latitude", "first"),
        longitude=("longitude", "first"),
        terrain_latitude=("terrain_reference_latitude", "first"),
        terrain_longitude=("terrain_reference_longitude", "first"),
        unique_rainfall_values=("rainfall_1hr_mm", "nunique"),
        max_hourly_rainfall=("rainfall_1hr_mm", "max")
    ).round(3)
)

# Inspect 2018 rainfall
print("\n2018 MONTHLY RAINFALL:")
df["year"] = df["timestamp"].dt.year
df["month"] = df["timestamp"].dt.month

print(
    df[df["year"] == 2018]
    .groupby("month")["rainfall_1hr_mm"]
    .agg(["sum", "max", "mean"])
    .round(3)
)


LOCALITY COORDINATES AND RAINFALL SUMMARY:
            latitude  longitude  terrain_latitude  terrain_longitude  \
zone_name                                                              
Andheri       19.120     72.846             19.12              72.85   
Bandra        19.054     72.841             19.06              72.85   
Borivali      19.231     72.857             19.23              72.85   
Goregaon      19.166     72.853             19.16              72.85   
Jogeshwari    19.137     72.849             19.14              72.85   
Kandivali     19.204     72.851             19.20              72.85   
Malad         19.187     72.848             19.18              72.85   
Santacruz     19.081     72.841             19.08              72.84   
Vile Parle    19.100     72.843             19.10              72.85   

            unique_rainfall_values  max_hourly_rainfall  
zone_name                                                
Andheri                        187              

In [18]:
import json

notebook_path = "extract_mumbai_weather.py.ipynb"

with open(notebook_path, "r", encoding="utf-8") as f:
    notebook = json.load(f)

for i, cell in enumerate(notebook["cells"]):
    source = "".join(cell.get("source", []))

    if "archive-api.open-meteo.com" in source or "models" in source:
        print(f"\n--- CELL {i} ---")
        print(source[:4000])


--- CELL 3 ---
API_URL = "https://archive-api.open-meteo.com/v1/archive"

START_DATE = "2015-01-01"
END_DATE = "2025-12-31"

OUTPUT_FILE = "mumbai_raw_weather_data.csv"

locations = {
    "Bandra": (19.0544, 72.8406),
    "Santacruz": (19.0810, 72.8410),
    "Vile Parle": (19.1000, 72.8430),
    "Andheri": (19.1197, 72.8464),
    "Jogeshwari": (19.1370, 72.8490),
    "Goregaon": (19.1663, 72.8526),
    "Malad": (19.1874, 72.8484),
    "Kandivali": (19.2045, 72.8510),
    "Borivali": (19.2307, 72.8567)
}

hourly_variables = [
    "precipitation",
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "weather_code"
]

# -----------------------------------
# 1. LOAD EXISTING DATA
# -----------------------------------

if os.path.exists(OUTPUT_FILE):
    final_df = pd.read_csv(OUTPUT_FILE)

    completed_locations = set(final_df["zone_name"].unique())

    print("Existing CSV loaded.")
    print("Already extracted:", completed_locations)

else:
    final_df = pd.Data

In [19]:
import requests

API_URL = "https://archive-api.open-meteo.com/v1/archive"

test_locations = {
    "Bandra": (19.0544, 72.8406),
    "Borivali": (19.2307, 72.8567),
    "Goregaon": (19.1663, 72.8526)
}

params = {
    "latitude": 19.0544,
    "longitude": 72.8406,
    "start_date": "2020-07-01",
    "end_date": "2020-07-03",
    "hourly": "precipitation",
    "timezone": "Asia/Kolkata"
}

for name, (lat, lon) in test_locations.items():
    params["latitude"] = lat
    params["longitude"] = lon

    response = requests.get(API_URL, params=params, timeout=60)
    response.raise_for_status()

    data = response.json()

    print(f"\n{name}")
    print("Requested:", lat, lon)
    print("Returned grid:", data.get("latitude"), data.get("longitude"))
    print("Elevation:", data.get("elevation"))
    print("Model:", data.get("model", "Not provided"))
    print("Rainfall:", data["hourly"]["precipitation"][:5])
    


Bandra
Requested: 19.0544 72.8406
Returned grid: 19.086115 72.85291
Elevation: 8.0
Model: Not provided
Rainfall: [0.4, 0.3, 0.4, 0.6, 0.4]

Borivali
Requested: 19.2307 72.8567
Returned grid: 19.226713 72.81899
Elevation: 19.0
Model: Not provided
Rainfall: [0.4, 0.4, 0.5, 0.4, 0.2]

Goregaon
Requested: 19.1663 72.8526
Returned grid: 19.156414 72.83597
Elevation: 21.0
Model: Not provided
Rainfall: [0.4, 0.4, 0.3, 0.7, 0.3]


# identify where the duplication entered

In [20]:
import pandas as pd
import hashlib

files = {
    "RAW": "mumbai_raw_weather_data.csv",
    "CLEAN": "mumbai_clean_weather_data_final.csv"
}

for label, path in files.items():
    data = pd.read_csv(path)
    data["timestamp"] = pd.to_datetime(data["timestamp"])

    print(f"\n{'='*45}\n{label} DATA")
    print("Shape:", data.shape)

    hashes = {}

    for zone, group in data.groupby("zone_name"):
        group = group.sort_values("timestamp")

        rainfall = group["rainfall_1hr_mm"].reset_index(drop=True)

        hashes[zone] = hashlib.sha256(
            pd.util.hash_pandas_object(
                rainfall, index=False
            ).values.tobytes()
        ).hexdigest()

    print("Unique rainfall sequences:", len(set(hashes.values())))

    for zone, fingerprint in hashes.items():
        matches = [
            other for other, other_fp in hashes.items()
            if other != zone and other_fp == fingerprint
        ]
        if matches:
            print(zone, "=", matches)


RAW DATA
Shape: (867888, 9)
Unique rainfall sequences: 3
Andheri = ['Bandra', 'Santacruz', 'Vile Parle']
Bandra = ['Andheri', 'Santacruz', 'Vile Parle']
Borivali = ['Kandivali']
Goregaon = ['Jogeshwari', 'Malad']
Jogeshwari = ['Goregaon', 'Malad']
Kandivali = ['Borivali']
Malad = ['Goregaon', 'Jogeshwari']
Santacruz = ['Andheri', 'Bandra', 'Vile Parle']
Vile Parle = ['Andheri', 'Bandra', 'Santacruz']

CLEAN DATA
Shape: (867465, 13)
Unique rainfall sequences: 3
Andheri = ['Bandra', 'Santacruz', 'Vile Parle']
Bandra = ['Andheri', 'Santacruz', 'Vile Parle']
Borivali = ['Kandivali']
Goregaon = ['Jogeshwari', 'Malad']
Jogeshwari = ['Goregaon', 'Malad']
Kandivali = ['Borivali']
Malad = ['Goregaon', 'Jogeshwari']
Santacruz = ['Andheri', 'Bandra', 'Vile Parle']
Vile Parle = ['Andheri', 'Bandra', 'Santacruz']


In [21]:
import pandas as pd

df = pd.read_csv("mumbai_raw_weather_data.csv")
df["timestamp"] = pd.to_datetime(df["timestamp"])

test_dates = [
    "2020-07-01 00:00:00",
    "2020-07-01 01:00:00",
    "2020-07-01 02:00:00",
    "2020-07-02 00:00:00"
]

for date in test_dates:
    print(f"\n--- {date} ---")

    result = df[df["timestamp"].isin(pd.to_datetime(test_dates))]
    result = result[result["timestamp"] == pd.Timestamp(date)]

    print(
        result[["zone_name", "timestamp", "rainfall_1hr_mm"]]
        .sort_values("zone_name")
        .to_string(index=False)
    )


--- 2020-07-01 00:00:00 ---
 zone_name  timestamp  rainfall_1hr_mm
   Andheri 2020-07-01              0.4
    Bandra 2020-07-01              0.4
  Borivali 2020-07-01              0.4
  Goregaon 2020-07-01              0.4
Jogeshwari 2020-07-01              0.4
 Kandivali 2020-07-01              0.4
     Malad 2020-07-01              0.4
 Santacruz 2020-07-01              0.4
Vile Parle 2020-07-01              0.4

--- 2020-07-01 01:00:00 ---
 zone_name           timestamp  rainfall_1hr_mm
   Andheri 2020-07-01 01:00:00              0.3
    Bandra 2020-07-01 01:00:00              0.3
  Borivali 2020-07-01 01:00:00              0.4
  Goregaon 2020-07-01 01:00:00              0.4
Jogeshwari 2020-07-01 01:00:00              0.4
 Kandivali 2020-07-01 01:00:00              0.4
     Malad 2020-07-01 01:00:00              0.4
 Santacruz 2020-07-01 01:00:00              0.3
Vile Parle 2020-07-01 01:00:00              0.3

--- 2020-07-01 02:00:00 ---
 zone_name           timestamp  rainfall_1h

In [22]:
import requests

locations = {
    "Bandra": (19.0544, 72.8406),
    "Santacruz": (19.0810, 72.8410),
    "Vile Parle": (19.1000, 72.8430),
    "Andheri": (19.1197, 72.8464),
    "Jogeshwari": (19.1370, 72.8490),
    "Goregaon": (19.1663, 72.8526),
    "Malad": (19.1874, 72.8484),
    "Kandivali": (19.2045, 72.8510),
    "Borivali": (19.2307, 72.8567)
}

url = "https://archive-api.open-meteo.com/v1/archive"

for name, (lat, lon) in locations.items():
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": "2020-07-01",
        "end_date": "2020-07-03",
        "hourly": "precipitation",
        "timezone": "Asia/Kolkata"
    }

    response = requests.get(url, params=params, timeout=60)
    response.raise_for_status()
    data = response.json()

    print(
        f"{name:12} | "
        f"Requested: {lat:.4f}, {lon:.4f} | "
        f"Grid: {data['latitude']:.5f}, {data['longitude']:.5f}"
    )

Bandra       | Requested: 19.0544, 72.8406 | Grid: 19.08611, 72.85291
Santacruz    | Requested: 19.0810, 72.8410 | Grid: 19.08611, 72.85291
Vile Parle   | Requested: 19.1000, 72.8430 | Grid: 19.08611, 72.85291
Andheri      | Requested: 19.1197, 72.8464 | Grid: 19.08611, 72.85291
Jogeshwari   | Requested: 19.1370, 72.8490 | Grid: 19.15641, 72.83597
Goregaon     | Requested: 19.1663, 72.8526 | Grid: 19.15641, 72.83597
Malad        | Requested: 19.1874, 72.8484 | Grid: 19.15641, 72.83597
Kandivali    | Requested: 19.2045, 72.8510 | Grid: 19.22671, 72.81899
Borivali     | Requested: 19.2307, 72.8567 | Grid: 19.22671, 72.81899


# final check wheter the dataset is clean

In [23]:
import pandas as pd
import numpy as np

# Load dataset
df = pd.read_csv("mumbai_weather_terrain_combined.csv")

print("=" * 60)
print("1. DATASET OVERVIEW")
print("=" * 60)

print("Shape:", df.shape)
print("Columns:", df.columns.tolist())

# 1. Missing values
print("\n2. MISSING VALUES")
missing = df.isnull().sum()
print(missing[missing > 0] if missing.any() else "No missing values")

# 2. Duplicate records
print("\n3. DUPLICATE CHECK")
print("Completely duplicated rows:", df.duplicated().sum())
print(
    "Duplicate locality-timestamp pairs:",
    df.duplicated(subset=["zone_name", "timestamp"]).sum()
)

# 3. Data types
print("\n4. DATA TYPES")
print(df.dtypes)

# 4. Timestamp validation
print("\n5. TIMESTAMP VALIDATION")
df["timestamp"] = pd.to_datetime(df["timestamp"], errors="coerce")
print("Invalid timestamps:", df["timestamp"].isna().sum())
print("Date range:", df["timestamp"].min(), "to", df["timestamp"].max())

# 5. Locality coverage
print("\n6. LOCALITY COVERAGE")
print(df["zone_name"].value_counts().sort_index())

# 6. Numeric invalid values
print("\n7. NUMERIC VALIDATION")
numeric_cols = df.select_dtypes(include=np.number).columns

print("Infinite values:")
print(np.isinf(df[numeric_cols]).sum().sum())

non_negative_cols = [
    "rainfall_1hr_mm", "rainfall_3hr_mm",
    "rainfall_6hr_mm", "rainfall_24hr_mm",
    "rainfall_48hr_mm", "relative_humidity_2m",
    "wind_speed_10m", "distance_to_coast_m",
    "distance_to_nearest_waterway_m"
]

for col in non_negative_cols:
    if col in df.columns:
        print(f"{col}: {(df[col] < 0).sum()} negative values")

if "relative_humidity_2m" in df.columns:
    print(
        "Humidity above 100:",
        (df["relative_humidity_2m"] > 100).sum()
    )

# 7. Chronological continuity
print("\n8. HOURLY CONTINUITY")
for zone, group in df.groupby("zone_name"):
    timestamps = group["timestamp"].sort_values()
    gaps = timestamps.diff().dropna()
    print(
        f"{zone}:",
        "Missing hourly intervals =", (gaps > pd.Timedelta(hours=1)).sum()
    )

# 8. Rainfall consistency
print("\n9. ROLLING RAINFALL CONSISTENCY")
for short, long in [
    ("rainfall_1hr_mm", "rainfall_3hr_mm"),
    ("rainfall_3hr_mm", "rainfall_6hr_mm"),
    ("rainfall_6hr_mm", "rainfall_24hr_mm"),
    ("rainfall_24hr_mm", "rainfall_48hr_mm")
]:
    if short in df.columns and long in df.columns:
        violations = (df[long] + 0.01 < df[short]).sum()
        print(f"{long} < {short}: {violations}")

print("\n" + "=" * 60)
print("AUDIT COMPLETED")
print("=" * 60)

1. DATASET OVERVIEW
Shape: (867465, 19)
Columns: ['zone_name', 'timestamp', 'rainfall_1hr_mm', 'temperature_2m', 'relative_humidity_2m', 'wind_speed_10m', 'weather_code', 'latitude', 'longitude', 'rainfall_3hr_mm', 'rainfall_6hr_mm', 'rainfall_24hr_mm', 'rainfall_48hr_mm', 'zone_id', 'terrain_reference_latitude', 'terrain_reference_longitude', 'elevation_m', 'distance_to_coast_m', 'distance_to_nearest_waterway_m']

2. MISSING VALUES
No missing values

3. DUPLICATE CHECK
Completely duplicated rows: 0
Duplicate locality-timestamp pairs: 0

4. DATA TYPES
zone_name                          object
timestamp                          object
rainfall_1hr_mm                   float64
temperature_2m                    float64
relative_humidity_2m                int64
wind_speed_10m                    float64
weather_code                        int64
latitude                          float64
longitude                         float64
rainfall_3hr_mm                   float64
rainfall_6hr_mm       